P9 - ANALYSE DE VENTES LIBRAIRIE

1 - Import et chargement des données

In [1]:
#import des librairies

import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# import des datasets
df_customers = pd.read_csv("../Data/customers.csv", sep=';')
df_products = pd.read_csv("../Data/products.csv", sep=';')
df_transactions = pd.read_csv("../Data/Transactions.csv", sep=';', engine='python')


# Aperçu rapide
print("Customers :")
display(df_customers.head())

print("Products :")
display(df_products.head())

print("Transactions :")
display(df_transactions.head())

Customers :


,client_id,sex,birth
0,c_4410,f,1967
1,c_7839,f,1975
2,c_1699,f,1984
3,c_5961,f,1962
4,c_5320,m,1943


Products :


,id_prod,price,categ
0,0_1421,19.99,0
1,0_1368,5.13,0
2,0_731,17.99,0
3,1_587,4.99,1
4,0_1507,3.99,0


Transactions :


,id_prod,date,session_id,client_id
0,0_1259,2021-03-01 00:01:07.843138,s_1,c_329
1,0_1390,2021-03-01 00:02:26.047414,s_2,c_664
2,0_1352,2021-03-01 00:02:38.311413,s_3,c_580
3,0_1458,2021-03-01 00:04:54.559692,s_4,c_7912
4,0_1358,2021-03-01 00:05:18.801198,s_5,c_2033


2 - Exploration et Nettoyage des données

In [2]:
#Identifier les Valeurs manquantes

print("\nValeurs manquantes :")
print("Customers :")
display(df_customers.isna().sum())

print("\nProducts :")
display(df_products.isna().sum())

print("\nTransactions :")
display(df_transactions.isna().sum())



Valeurs manquantes :
Customers :


client_id    0
sex          0
birth        0
dtype: int64


Products :


id_prod    0
price      0
categ      0
dtype: int64


Transactions :


id_prod       0
date          0
session_id    0
client_id     0
dtype: int64

In [3]:
#Suppression des lignes totalement vides
df_transactions = df_transactions.dropna(how='all')

#Suppression des lignes où les infos essentielles manquent
df_transactions = df_transactions.dropna(subset=['date', 'session_id', 'client_id'])

#Verification des valeurs manquantes après nettoyage
df_transactions.isna().sum()
df_transactions.duplicated().sum()

np.int64(0)

In [4]:
#Vérification des Doublons

print("\nDoublons :")
print("Customers :", df_customers.duplicated().sum())
print("Products :", df_products.duplicated().sum())
print("Transactions :", df_transactions.duplicated().sum())

#Nettoyage des types
#Conversion de la date en datetime
df_transactions['date'] = pd.to_datetime(df_transactions['date'], errors='coerce')


# Vérification
df_transactions['date'].head()


Doublons :
Customers : 0
Products : 0


Transactions : 0


0   2021-03-01 00:01:07.843138
1   2021-03-01 00:02:26.047414
2   2021-03-01 00:02:38.311413
3   2021-03-01 00:04:54.559692
4   2021-03-01 00:05:18.801198
Name: date, dtype: datetime64[us]

3- Construction du dataset final

In [5]:
#Merge transactions + products
df_sales = df_transactions.merge(df_products, on="id_prod", how="left")

#Merge avec customers
df_sales = df_sales.merge(df_customers, on="client_id",how="left")

#Vérification
print("Taille du dataset final :", df_sales.shape)
df_sales.head()

Taille du dataset final : (687534, 8)


,id_prod,date,session_id,client_id,price,categ,sex,birth
0,0_1259,2021-03-01 00:01:07.843138,s_1,c_329,11.99,0,f,1967
1,0_1390,2021-03-01 00:02:26.047414,s_2,c_664,19.37,0,m,1960
2,0_1352,2021-03-01 00:02:38.311413,s_3,c_580,4.50,0,m,1988
3,0_1458,2021-03-01 00:04:54.559692,s_4,c_7912,6.55,0,f,1989
4,0_1358,2021-03-01 00:05:18.801198,s_5,c_2033,16.49,0,f,1956


In [6]:
#Ajout de la colonne chiffre d'affaires
df_sales["chiffre_affaires"] = df_sales["price"]

#Extraction des informations temporelles
df_sales["year"] = df_sales["date"].dt.year
df_sales["month"] = df_sales["date"].dt.month
df_sales["day"] = df_sales["date"].dt.day
df_sales["hour"] = df_sales["date"].dt.hour

4- Analyse du CA

In [7]:
#Calcul du chiffre d'affaires total
chiffre_affaires_total = df_sales["chiffre_affaires"].sum()
print("Chiffre d'affaires total :", round(chiffre_affaires_total, 2))

#Chiffre d'affaires par mois
chiffre_affaires_mensuel = (df_sales.groupby(["year", "month"])["chiffre_affaires"].sum().round(2).reset_index())

#Création d'une date (1er du mois)
chiffre_affaires_mensuel["date"] = pd.to_datetime(chiffre_affaires_mensuel["year"].astype(str) + "-" + chiffre_affaires_mensuel["month"].astype(str) + "-01")

chiffre_affaires_mensuel.head()

Chiffre d'affaires total : 12027663.1


,year,month,chiffre_affaires,date
0,2021,3,482440.61,2021-03-01
1,2021,4,476109.30,2021-04-01
2,2021,5,492943.47,2021-05-01
3,2021,6,484088.56,2021-06-01
4,2021,7,482835.40,2021-07-01


In [8]:
#Chiffre d'affaires par catégorie
chiffre_affaires_categ = df_sales.groupby("categ")["chiffre_affaires"].sum().round(2).sort_values(ascending=False)
chiffre_affaires_categ


categ
1    4827657.11
0    4419730.97
2    2780275.02
Name: chiffre_affaires, dtype: float64

In [9]:
#Chiffre d'affaires par produit
chiffre_affaires_prod = df_sales.groupby("id_prod")["chiffre_affaires"].sum().round(2).sort_values(ascending=False)
chiffre_affaires_prod.head()

id_prod
2_159    94893.50
2_135    69334.95
2_112    65407.76
2_102    60736.78
2_209    56971.86
Name: chiffre_affaires, dtype: float64

In [10]:
#Chiffre d'affaires par client
chiffre_affaires_client = df_sales.groupby("client_id")["chiffre_affaires"].sum().round(2).sort_values(ascending=False)
chiffre_affaires_client.head()

client_id
c_1609    326039.89
c_4958    290227.03
c_6714    153918.60
c_3454    114110.57
c_1570      5285.82
Name: chiffre_affaires, dtype: float64

In [11]:
#CA par jour de la semaine et par heure dans le but de créer une heatmap
df_sales["weekday"] = df_sales["date"].dt.day_name()

heatmap_data = df_sales.groupby(["weekday", "hour"])["chiffre_affaires"].sum().unstack().round(2)

heatmap_data

hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
weekday,,,,,,,,,,,,,,,,,,,,,
Friday,71310.30,68940.53,69122.59,70857.62,70022.08,71261.39,73997.59,75286.09,75285.00,69407.50,...,70396.46,71566.92,68847.03,68421.93,71268.12,70772.15,70209.32,68488.97,71168.52,73353.02
Monday,71434.39,70917.27,75730.85,72528.93,70094.64,68852.58,74838.77,69859.24,69966.24,73506.02,...,73255.45,68461.29,73026.02,73774.37,74569.09,72325.62,71396.25,76091.44,71596.11,72712.51
Saturday,71705.66,72470.10,71039.60,69226.76,71042.83,67945.94,68990.92,68155.80,73094.30,71413.70,...,74647.87,70230.12,71147.76,72191.59,68730.74,71475.04,70296.10,69658.43,71555.39,70304.24
Sunday,70202.57,71431.96,73419.42,70412.93,73126.13,69832.39,71527.28,71151.00,72953.61,71235.40,...,71564.89,69927.41,71503.45,72174.02,77040.55,74508.68,69347.51,71534.14,71194.10,71953.37
Thursday,68313.80,71270.76,72758.94,69953.23,71935.73,69111.26,70747.40,70440.84,73853.39,70673.31,...,69756.50,71323.65,70947.17,71322.90,72699.12,70765.07,71278.63,71425.54,71644.64,73799.34
Tuesday,70405.25,72359.05,73164.40,71283.04,71720.05,72990.60,73064.78,73135.70,71194.60,72742.38,...,72076.22,70959.61,71125.02,72258.90,72945.85,71302.03,71311.49,73051.42,70641.01,79559.41
Wednesday,67097.42,66978.42,74521.18,71064.60,72849.83,70187.16,71218.95,71754.46,69778.24,72301.49,...,69901.20,70377.93,70798.84,70791.09,73662.77,72666.08,69209.48,71095.55,72677.28,75079.48


5 - Visualisations des analyses de ventes

In [12]:
#Graphique de l'évolution du CA dans le temps
import plotly.express as px

fig = px.line(
    chiffre_affaires_mensuel,
    x="month",
    y="chiffre_affaires",
    color="year",
    markers=True,
    title="Chiffre d'affaires mensuel (par année)"
)

fig.update_traces(hovertemplate="Année: %{fullData.name}<br>Mois: %{x}<br>CA: %{y} €")

fig.update_layout(
    xaxis_title="Mois",
    yaxis_title="CA (€)"
)

fig.show()



Le chiffre d'affaires mensuel est relativement stable sur la période observée. Sur les mois comparables de mars à décembre, le CA de 2022 dépasse celui de 2021 d'environ 2,1 %. Les données ne suffisent pas à établir un pic récurrent des fêtes. En 2023, seuls janvier et février sont disponibles et février montre une baisse.


In [13]:
#CA et moyenne mobile
chiffre_affaires_mensuel = (df_sales.groupby(["year", "month"])["chiffre_affaires"].sum().reset_index())

#Création d'une vraie date
chiffre_affaires_mensuel["date"] = pd.to_datetime(chiffre_affaires_mensuel["year"].astype(str) + "-" + chiffre_affaires_mensuel["month"].astype(str) + "-01")


#Moyenne mobile 3 mois (sur toute la série continue)
chiffre_affaires_mensuel = chiffre_affaires_mensuel.sort_values("date")
chiffre_affaires_mensuel["MA3"] = chiffre_affaires_mensuel["chiffre_affaires"].rolling(3).mean()

import plotly.graph_objects as go

fig = go.Figure()

#Ligne CA
fig.add_trace(go.Scatter(
    x=chiffre_affaires_mensuel["date"],
    y=chiffre_affaires_mensuel["chiffre_affaires"],
    mode="lines+markers",
    name="CA mensuel",
    line=dict(color="#1f77b4", width=2),
    hovertemplate="Date: %{x}<br>CA: %{y} €"
))

#Ligne moyenne mobile
fig.add_trace(go.Scatter(
    x=chiffre_affaires_mensuel["date"],
    y=chiffre_affaires_mensuel["MA3"],
    mode="lines+markers",
    name="Moyenne mobile (3 mois)",
    line=dict(color="#d62728", width=2, dash="dash"),
    hovertemplate="Date: %{x}<br>MA3: %{y} €"
))

fig.update_layout(
    title="CA mensuel + moyenne mobile (3 mois)",
    xaxis_title="Date",
    yaxis_title="CA (€)",
    hovermode="x unified"
)

fig.show()



La moyenne mobile sur trois mois atténue les variations mensuelles. Le graphique montre une progression limitée entre les périodes comparables de 2021 et 2022, puis une baisse en février 2023. Deux années de données ne permettent pas d'établir une saisonnalité récurrente.


In [14]:
#CA par catégorie sous forme de graphique

#Calcul du CA par catégorie et du pourcentage
ca_categ = df_sales.groupby("categ")["chiffre_affaires"].sum().reset_index()
ca_categ["pct"] = ca_categ["chiffre_affaires"] / ca_categ["chiffre_affaires"].sum() * 100

# Format CA avec séparateurs de milliers
ca_categ["ca_fmt"] = ca_categ["chiffre_affaires"].apply(lambda x: f"{x:,.0f}".replace(",", " "))
ca_categ_sorted = ca_categ.sort_values("chiffre_affaires", ascending=False)

# Conversion de la colonne "categ" en chaîne de caractères pour l'affichage
ca_categ_sorted["categ_str"] = ca_categ_sorted["categ"].astype(str)

import plotly.express as px

fig = px.bar(
    ca_categ_sorted,
    x="chiffre_affaires",
    y="categ_str",
    orientation="h",
    text=ca_categ_sorted["ca_fmt"] + " €" + "<br>" + ca_categ_sorted["pct"].round(1).astype(str) + " %",
    title="CA par catégorie (CA + %)",
    color="categ_str",
    color_discrete_sequence=["#1f77b4", "#ff7f0e", "#2ca02c"]
)

fig.update_traces(
    hovertemplate="Catégorie: %{y}<br>CA: %{x:,.0f} €<br>Part: %{customdata} %",
    customdata=ca_categ_sorted["pct"].round(1),
    textposition="outside",
    textfont=dict(size=14)
)

fig.update_layout(
    xaxis=dict(
        title="CA (€)",
        range=[0, 6_000_000],
    ),
    yaxis=dict(
        title="Catégorie"
    ),
    margin=dict(l=120, r=40, t=80, b=40)
)

fig.show()


Les catégories 0 et 1 génèrent la majorité du chiffre d’affaires, avec respectivement 36,7 % et 40,1 % du total. 
La catégorie 2 est moins performante (23,1 %), mais reste significative. 
La répartition montre un portefeuille équilibré, dominé par deux catégories principales.

In [15]:
#Nombre de clients par mois
clients_mois = df_sales.groupby(["year", "month"])["client_id"].nunique().reset_index()
clients_mois.rename(columns={"client_id": "nb_clients"}, inplace=True)

#Graphique de l'évolution du nombre de clients uniques par mois
fig = px.line(
    clients_mois,
    x="month",
    y="nb_clients",
    color="year",
    markers=True,
    title="Nombre de clients uniques par mois (par année)"
)

fig.update_traces(hovertemplate="Année: %{fullData.name}<br>Mois: %{x}<br>Clients: %{y}")
fig.update_layout(xaxis_title="Mois", yaxis_title="Clients uniques")
fig.show()



Le nombre de clients uniques varie selon les années, avec un pic marqué en 2021 autour du mois d’octobre. 
L’année 2022 est plus stable mais légèrement inférieure en volume.
En 2023, le mois de février montre une baisse, suggérant un début d’année moins dynamique en termes d’acquisition ou de fidélisation.

In [16]:
#Nombre de transactions par mois
transactions_mois = (df_sales.groupby(["year", "month"])["session_id"].nunique().reset_index())

transactions_mois.rename(columns={"session_id": "nb_transactions"}, inplace=True)

#Graphique du nombre de transactions par mois
fig = px.line(
    transactions_mois,
    x="month",
    y="nb_transactions",
    color="year",
    markers=True,
    title="Nombre de transactions par mois (par année)"
)

fig.update_traces(hovertemplate="Année: %{fullData.name}<br>Mois: %{x}<br>Transactions: %{y}")

fig.update_layout(xaxis_title="Mois",yaxis_title="Nombre de transactions")

fig.show()



Le volume de transactions suit une dynamique similaire aux ventes : 2021 est plus volatile, 2022 plus homogène, et 2023 démarre en baisse. 
Cette tendance suggère une diminution de la fréquentation ou de l’activité d’achat en début d’année 2023.

In [17]:
#Nombre de produits vendus par mois
produits_mois = (
    df_sales
    .groupby(["year", "month"])["id_prod"]
    .count()
    .reset_index()
)

produits_mois.rename(columns={"id_prod": "nb_produits"}, inplace=True)

#Graphique du nombre de produits vendus par mois
fig = px.line(
    produits_mois,
    x="month",
    y="nb_produits",
    color="year",
    markers=True,
    title="Nombre de produits vendus par mois (par année)"
)

fig.update_traces(hovertemplate="Année: %{fullData.name}<br>Mois: %{x}<br>Produits vendus: %{y}")

fig.update_layout(
    xaxis_title="Mois",
    yaxis_title="Produits vendus"
)

fig.show()



Les ventes de produits présentent une forte variabilité en 2021, avec un pic notable à la rentrée. 
L’année 2022 est plus régulière, traduisant une activité plus stable. 
En 2023, les données partielles montrent un début d’année en recul, cohérent avec la baisse observée sur les autres indicateurs.

In [18]:
#Top 10 des produits les plus vendus
top10 = df_sales.groupby("id_prod")["chiffre_affaires"].sum().sort_values(ascending=False).head(10).reset_index()
top10_sorted = top10.sort_values("chiffre_affaires", ascending=True)

#Graphique du top 10 des produits les plus vendus
fig = px.bar(
    top10_sorted,
    x="chiffre_affaires",
    y="id_prod",
    orientation="h",
    title="Top 10 références par CA",
    color="chiffre_affaires"
)

fig.update_traces(hovertemplate="Produit: %{y}<br>CA: %{x:,.0f} €")

fig.update_layout(
    xaxis_title="CA (€)",
    yaxis_title="ID Produit"
)

fig.show()



Les dix premières références représentent 5,17 % du chiffre d'affaires total. Six appartiennent à la catégorie 2. Le classement porte sur le CA et ne mesure pas la rentabilité, faute de coûts et de marges.


In [19]:
#Flop 10 des produits les moins vendus
flop10 = df_sales.groupby("id_prod")["chiffre_affaires"].sum().sort_values().head(10).reset_index()

fig = px.bar(
    flop10,
    x="chiffre_affaires",
    y="id_prod",
    orientation="h",
    title="Flop 10 références par CA",
    color="chiffre_affaires"
)

fig.update_traces(hovertemplate="Produit: %{y}<br>CA: %{x} €")
fig.update_layout(xaxis_title="CA (€)", yaxis_title="ID Produit")
fig.show()


Ces dix références génèrent chacune très peu de chiffre d'affaires sur la période. Les données disponibles ne permettent pas de conclure sur leur marge, leur stock ou l'intérêt d'un déréférencement.


In [20]:
#Mise en ordre des jours de la semaine pour la heatmap
ordre_jours = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heatmap_data = heatmap_data.reindex(ordre_jours)


#Heatmap du CA par jour de la semaine et par heure
import plotly.graph_objects as go
fig = px.imshow(
    heatmap_data,
    labels=dict(x="Heure", y="Jour", color="CA (€)"),
    title="Heatmap du CA par jour et par heure",
    color_continuous_scale="YlOrRd"
)

fig.update_traces(
    hovertemplate=
        "Jour: %{y}<br>" +
        "Heure: %{x}h<br>" +
        "CA: %{z:,.0f} €"
)

#Remplacer virgules US par espaces FR
fig.update_traces(
    hovertemplate=fig.data[0].hovertemplate.replace(",", " ")
)

fig.update_layout(
    xaxis_title="Heure",
    yaxis_title="Jour"
)

fig.show()


Le chiffre d'affaires total par heure varie peu sur la période. Cette heatmap ne met pas en évidence un pic stable à midi ; elle montre surtout des écarts limités selon le jour et l'heure.


6- Analyses et visualistaions de la clientèle

In [21]:
# Les analyses par âge et genre portent uniquement sur les clients BtoC.
# Les indicateurs commerciaux globaux ci-dessus conservent les quatre comptes BtoB.
liste_b2b = ["c_1609", "c_4958", "c_6714", "c_3454"]
df_final = df_sales.loc[~df_sales["client_id"].isin(liste_b2b)].copy()
df_final["age"] = 2022 - df_final["birth"].astype(int)


In [22]:
# Chiffre d'affaires par genre et catégorie chez les clients BtoC.
genre_categ = (
    df_final.groupby(["sex", "categ"], observed=True)["chiffre_affaires"]
    .sum().reset_index()
)
fig = px.bar(
    genre_categ,
    x="categ",
    y="chiffre_affaires",
    color="sex",
    barmode="group",
    title="CA BtoC par catégorie selon le genre",
    labels={"categ": "Catégorie", "chiffre_affaires": "CA (€)", "sex": "Genre"},
)
fig.show()


Le graphique compare le chiffre d'affaires BtoC par genre et catégorie. Il décrit des volumes de vente et ne mesure pas une préférence individuelle : les groupes peuvent comporter des nombres de clients différents.


In [23]:
# Tranches d'âge avec bornes correspondant aux étiquettes.
df_final["age_group"] = pd.cut(
    df_final["age"],
    bins=[0, 19, 29, 39, 49, 59, 69, 200],
    labels=["0-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70+"],
    right=True,
)

age_group_ca = (
    df_final.groupby("age_group", observed=True)["chiffre_affaires"]
    .sum().round(2).reset_index()
)

fig = px.bar(
    age_group_ca,
    x="age_group",
    y="chiffre_affaires",
    title="Montant total des achats par tranche d'âge",
    labels={"age_group": "Tranche d'âge", "chiffre_affaires": "Montant total des achats (€)"},
    color="age_group",
)
fig.update_layout(showlegend=False, xaxis_title="Tranche d'âge", yaxis_title="Montant total des achats (€)")
fig.show()


Chez les clients BtoC, les 30–49 ans contribuent fortement au chiffre d'affaires par tranche d'âge. Ces totaux dépendent aussi du nombre de clients dans chaque tranche ; ils ne mesurent pas le pouvoir d'achat individuel.


In [24]:
# Une fréquence par client : nombre de sessions d'achat distinctes.
freq_achats = (
    df_final.groupby("client_id")
    .agg(freq_achats=("session_id", "nunique"), age_group=("age_group", "first"))
    .reset_index()
)
freq_age_group = (
    freq_achats.groupby("age_group", observed=True)["freq_achats"]
    .median().reset_index()
)

fig = px.bar(
    freq_age_group,
    x="age_group",
    y="freq_achats",
    title="Nombre médian de sessions par client et par tranche d'âge",
    labels={"age_group": "Tranche d'âge", "freq_achats": "Sessions par client (médiane)"},
    color="age_group",
)
fig.update_layout(showlegend=False, xaxis_title="Tranche d'âge", yaxis_title="Sessions par client (médiane)")
fig.show()


La médiane du nombre de sessions par client est la plus élevée pour les 30–49 ans. Ce calcul compte chaque session une seule fois par client ; il ne multiplie plus la fréquence par le nombre de produits achetés.


In [25]:
#Calcul du panier moyen par client
client_panier = (df_final.groupby("client_id").agg({"chiffre_affaires": "sum", "session_id": "nunique", "age_group": "first"}).reset_index())
client_panier["panier_moyen"] = (client_panier["chiffre_affaires"] / client_panier["session_id"]).round(2)

#Panier moyen médian par tranche d'âge
panier_age_group = (client_panier.groupby("age_group", observed=True)["panier_moyen"].median().reset_index())

#Graphique du panier moyen médian par tranche d'âge
fig = px.bar(
    panier_age_group,
    x="age_group",
    y="panier_moyen",
    title="Panier moyen (médiane) par tranche d'âge",
    labels={"age_group": "Tranche d'âge", "panier_moyen": "Panier moyen (€)"},
    color="age_group"
)

fig.update_layout(
    showlegend=False,
    xaxis_title="Tranche d'âge",
    yaxis_title="Panier moyen (€)"
)

fig.show()

Chez les clients BtoC, la médiane du panier moyen par client est la plus élevée pour les moins de 30 ans. La fréquence des sessions est analysée séparément ; ces écarts descriptifs ne prouvent pas un effet causal de l'âge.


In [26]:
#CA par catégorie et tranche d'âge
age_categ = (df_final.groupby(["age_group", "categ"], observed=True)["chiffre_affaires"].sum().reset_index())

#Graphique multi-lignes du CA par catégorie et tranche d'âge
fig = px.line(
    age_categ,
    x="age_group",
    y="chiffre_affaires",
    color="categ",
    markers=True,
    title="Catégories de livres achetées selon la tranche d'âge",
    labels={
        "age_group": "Tranche d'âge",
        "chiffre_affaires": "Montant total des achats (€)",
        "categ": "Catégorie"
    }
)

fig.update_layout(
    xaxis_title="Tranche d'âge",
    yaxis_title="Montant total des achats (€)"
)

fig.show()


Le chiffre d'affaires BtoC par catégorie varie selon la tranche d'âge. Ces montants bruts dépendent des effectifs et des fréquences d'achat ; ils ne suffisent pas à établir des préférences individuelles pour une catégorie.


In [27]:
# Les quatre comptes BtoB sont identifiés dans la consigne du projet.
# Contrôle de leur poids dans le CA avant leur exclusion des analyses BtoC.
ca_clients = (
    df_sales.groupby("client_id")["chiffre_affaires"]
    .sum().round(2).sort_values(ascending=False).head(10).reset_index()
)
ca_clients


,client_id,chiffre_affaires
0,c_1609,326039.89
1,c_4958,290227.03
2,c_6714,153918.60
3,c_3454,114110.57
4,c_1570,5285.82
5,c_3263,5276.87
6,c_2140,5260.18
7,c_2899,5214.05
8,c_7319,5155.77
9,c_7959,5135.75


In [28]:
# Compter les sessions distinctes par client (et non additionner une
# fréquence répétée sur chaque ligne de produit).
freq_clients = (
    df_sales.groupby("client_id")["session_id"]
    .nunique().sort_values(ascending=False).head(10).reset_index(name="nb_sessions")
)
freq_clients


,client_id,nb_sessions
0,c_1609,10997
1,c_3454,5571
2,c_4958,3851
3,c_6714,2620
4,c_8526,167
5,c_2265,165
6,c_1637,165
7,c_669,163
8,c_8510,162
9,c_2077,161


In [29]:
# L'analyse de concentration porte sur les clients particuliers uniquement.
df_stats = df_final.copy()

ca_clients = (
    df_stats.groupby("client_id")["chiffre_affaires"]
    .sum().sort_values().reset_index()
)
n_clients = len(ca_clients)
ca_clients["client_cum"] = np.arange(1, n_clients + 1) / n_clients
ca_clients["ca_cum"] = ca_clients["chiffre_affaires"].cumsum() / ca_clients["chiffre_affaires"].sum()

part_clients = np.r_[0, ca_clients["client_cum"].to_numpy()]
part_ca = np.r_[0, ca_clients["ca_cum"].to_numpy()]
gini = 1 - 2 * np.trapezoid(part_ca, part_clients)
print("Coefficient de Gini (clients particuliers) :", round(gini, 3))

fig = px.line(
    x=part_clients,
    y=part_ca,
    title=f"Courbe de Lorenz du CA BtoC (Gini = {gini:.3f})",
    labels={"x": "Proportion cumulée de clients", "y": "Proportion cumulée du CA"},
)
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", name="Égalité parfaite", line=dict(color="black", dash="dash"))
fig.show()


Coefficient de Gini (clients particuliers) : 0.398


Après exclusion des quatre clients BtoB indiqués dans la consigne, l'indice de Gini d'environ 0,398 décrit une concentration modérée du chiffre d'affaires entre les clients particuliers. Cette mesure ne dit rien de la rentabilité des clients.
